In [5]:
import os
import re
import time
import unicodedata
import pandas as pd
import requests
from bs4 import BeautifulSoup

try:
    import ftfy  # canonical mojibake fixer; pip install ftfy
except ImportError:
    ftfy = None


def clean_text(s: str) -> str:
    """Repair encoding damage and tidy whitespace, but keep the article's REAL punctuation
    (curly quotes, en/em dashes) so downstream character offsets stay faithful to the source.
    This is the upstream fix that makes fix_mojibake()/_demojibake() elsewhere a safety net."""
    if not s:
        return s
    # 1. repair UTF-8-read-as-Latin-1 damage ("Â£" -> "£", "â€“" -> "–")
    if ftfy is not None:
        s = ftfy.fix_text(s)
    elif any(m in s for m in ("Ã", "Â", "â€")):
        try:
            s = s.encode("latin-1", "ignore").decode("utf-8", "ignore")
        except (UnicodeDecodeError, UnicodeEncodeError):
            pass
    # 2. canonical unicode form + strip invisible junk (NBSP, zero-width, soft hyphen)
    s = unicodedata.normalize("NFC", s)
    s = s.replace(" ", " ").replace("​", "").replace("­", "")
    # 3. tidy whitespace without touching real newlines between paragraphs
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r" *\n *", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


# --- Fetch strategy ---------------------------------------------------------------------------
# cbsnews.com (Akamai) bot-scores the caller and starts answering 406 to *every* request once it
# has flagged you — including ones that succeeded moments earlier. It is rate-driven, not header-
# driven, so the fix is: reuse one Session (keeps the clearance cookie), pause between articles,
# and back off on a challenge. Dropping the User-Agent entirely is the variant that reliably
# clears an active 406 block, so it is kept as the last-resort retry.
_UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)
_ACCEPT = "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8"

HEADER_VARIANTS = [
    {"User-Agent": _UA, "Accept": _ACCEPT, "Accept-Language": "en-US,en;q=0.9",
     "Upgrade-Insecure-Requests": "1"},
    # no User-Agent at all: an Akamai edge that is 406-ing the browser UA still serves this
    {"User-Agent": None, "Accept": _ACCEPT, "Accept-Language": "en-US,en;q=0.9"},
]

CHALLENGE_CODES = {403, 406, 429, 503}


def fetch(url, session=None, attempts=3, base_wait=5.0):
    """GET `url`, retrying across header variants with exponential backoff.

    Returns the successful Response. Raises the last error if every attempt fails."""
    session = session or requests.Session()
    last_exc = None
    for attempt in range(attempts):
        for headers in HEADER_VARIANTS:
            try:
                r = session.get(url, headers=headers, timeout=30)
                if r.status_code in CHALLENGE_CODES:
                    last_exc = requests.HTTPError(
                        f"{r.status_code} challenge for url: {url}", response=r
                    )
                    continue
                r.raise_for_status()
                return r
            except requests.RequestException as e:
                last_exc = e
        wait = base_wait * (2 ** attempt)
        print(f"      … blocked, backing off {wait:.0f}s before retry {attempt + 2}/{attempts}")
        time.sleep(wait)
    raise last_exc


def get_article_text(url, session=None):
    response = fetch(url, session=session)

    # Pass raw BYTES (not response.text): lets BeautifulSoup sniff the page's declared charset
    # instead of requests defaulting to ISO-8859-1 when the header omits it — that default is the
    # source of the "Â£3.74m" mojibake we were repairing downstream.
    soup = BeautifulSoup(response.content, "html.parser")

    paragraphs = soup.find_all("p")
    article_text = "\n".join(
        p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)
    )

    return clean_text(article_text)


def _existing_title(path):
    """First-line Title: of an already-scraped file, or None."""
    try:
        with open(path, encoding="utf-8") as f:
            first = f.readline()
    except OSError:
        return None
    return first[len("Title:"):].strip() if first.startswith("Title:") else None


def main(skip_existing=True, delay=3.0):
    """Scrape every row of articles.csv into articles/art{i+1}.txt.

    skip_existing=True leaves a file alone when its Title header already matches the CSV row,
    so a re-run only refetches what is missing or stale. This matters because some sources
    (usnews.com) that were reachable when first scraped now block us outright — refetching
    them unconditionally would trade good text for a failure."""
    df = pd.read_csv("articles.csv")

    os.makedirs("articles", exist_ok=True)

    total = len(df)
    session = requests.Session()
    failed = []

    for i, row in df.iterrows():
        title = row["title"]
        url = row["url"]
        date = row.get("Date", "")

        output_file = os.path.join("articles", f"art{i+1}.txt")

        print(f"[{i+1}/{total}] Processing: {title}")

        if skip_existing and _existing_title(output_file) == clean_text(title):
            print(f"    · Up to date, skipped ({output_file})")
            continue

        try:
            article_text = get_article_text(url, session=session)

            with open(output_file, "w", encoding="utf-8") as f:
                f.write(f"Title: {clean_text(title)}\n")
                f.write(f"URL: {url}\n")
                f.write(f"Date: {date}\n")
                f.write("\n")
                f.write(article_text)

            print(f"    ✓ Saved to {output_file} ({len(article_text)} characters)")

        except Exception as e:
            print(f"    ✗ Failed: {e}")
            failed.append((i + 1, title, str(e)))

        time.sleep(delay)  # stay under the rate threshold that triggers the 406 wall

    print("\nDone!")
    if failed:
        print(f"\n{len(failed)} row(s) failed — existing files for them were left untouched:")
        for n, t, e in failed:
            print(f"  row {n}: {t[:60]} — {e[:80]}")


if __name__ == "__main__":
    main()

[1/67] Processing: France records 2,025 excess deaths at peak of heatwave as Europe braces for more extreme weather
    · Up to date, skipped (articles/art1.txt)
[2/67] Processing: Taylor Swift's rumoured wedding celebrations begin with star-studded New York event
    · Up to date, skipped (articles/art2.txt)
[3/67] Processing: Plots, love letters and remedies: The medieval secrets being revealed by AI
    · Up to date, skipped (articles/art3.txt)
[4/67] Processing: What is US Open prize money?
    · Up to date, skipped (articles/art4.txt)
[5/67] Processing: Ebola treatments trial begins in DR Congo
    · Up to date, skipped (articles/art5.txt)
[6/67] Processing: Nagelsmann quits as Germany coach after early World Cup 2026 exit
    · Up to date, skipped (articles/art6.txt)
[7/67] Processing: Israel approves plan to establish 13 new settlements in occupied West Bank
    · Up to date, skipped (articles/art7.txt)
[8/67] Processing: ‘Never the same’: How war on Iran changed the global ener

KeyboardInterrupt: 

In [ ]:
# --- One-time cleanup of the EXISTING dataset/articles/*.txt (no re-scrape) -------------------
# Re-runs clean_text() over the files already on disk so current results are reproducible from
# clean text without re-fetching live pages (which may have changed). Idempotent: files already
# clean are left byte-identical. Run once; safe to re-run.
from pathlib import Path

art_dir = Path("articles")
changed = []
for f in sorted(art_dir.glob("*.txt")):
    original = f.read_text(encoding="utf-8")
    cleaned = clean_text(original)
    if cleaned != original:
        f.write_text(cleaned, encoding="utf-8")
        changed.append(f.name)

print(f"cleaned {len(changed)}/{len(list(art_dir.glob('*.txt')))} files"
      + (":  " + ", ".join(changed) if changed else "  (all already clean)"))


cleaned 0/19 files  (all already clean)


In [ ]:
# --- One-time backfill of the Date: header into EXISTING articles/*.txt (no re-scrape) --------
# The old scraper never wrote a Date line, so reference-date resolution (extract_e2e/_reference_date,
# which reads "^Date: dd/mm/yyyy") was getting None. Insert it from articles.csv, mapping row i ->
# art{i+1}.txt, right after the URL line. Idempotent: skips files that already have a header Date.
import pandas as pd
from pathlib import Path

df = pd.read_csv("articles.csv")
art_dir = Path("articles")
added = []
for i, row in df.iterrows():
    f = art_dir / f"art{i+1}.txt"
    if not f.exists():
        continue
    lines = f.read_text(encoding="utf-8").splitlines()
    # locate the URL header line; the Date line goes immediately after it
    url_idx = next((k for k, l in enumerate(lines[:5]) if l.startswith("URL:")), None)
    if url_idx is None:
        print(f"    ! {f.name}: no URL header line, skipped")
        continue
    if url_idx + 1 < len(lines) and lines[url_idx + 1].startswith("Date:"):
        continue  # already backfilled
    lines.insert(url_idx + 1, f"Date: {row.get('Date', '')}")
    f.write_text("\n".join(lines), encoding="utf-8")
    added.append(f.name)

print(f"backfilled Date into {len(added)} file(s)"
      + (":  " + ", ".join(added) if added else "  (all already had a Date header)"))


backfilled Date into 0 file(s)  (all already had a Date header)


In [ ]:
# --- Realign articles/art{N}.txt after articles.csv was edited (RUN BEFORE main()) ------------
# art{N}.txt is keyed by CSV *row position*, so inserting/removing/reordering a row silently
# renumbers every file after it — and every results/<model>/art{N}/ directory keeps pointing at
# whatever text used to sit in that slot. This cell re-derives the mapping from the Title header
# (which is content, not position), so files move to the slot their CSV row now occupies instead
# of being overwritten by a different article.
#
# Dry run by default. Set APPLY = True to perform the moves.
APPLY = False

import shutil
import pandas as pd
from pathlib import Path

art_dir = Path("articles")
archive_dir = art_dir / "_archive"
df = pd.read_csv("articles.csv")

files = sorted(art_dir.glob("art*.txt"), key=lambda p: int(p.stem[3:]))
titles = {f: _existing_title(f) for f in files}
wanted = {i + 1: clean_text(row["title"]) for i, row in df.iterrows()}

# Pass 1: a file already sitting in its correct slot keeps it. Without this, a duplicate copy
# elsewhere on disk could win the slot and evict the good file.
claimed, resolved = set(), {}
for n, title in wanted.items():
    here = art_dir / f"art{n}.txt"
    if titles.get(here) == title:
        claimed.add(here)
        resolved[n] = here

# Pass 2: every remaining row pulls its article from wherever that title currently lives.
missing = []
for n, title in wanted.items():
    if n in resolved:
        continue
    src = next((f for f in files if f not in claimed and titles.get(f) == title), None)
    if src is None:
        missing.append((n, title))
    else:
        claimed.add(src)
        resolved[n] = src

moves = [(src, art_dir / f"art{n}.txt", wanted[n])
         for n, src in sorted(resolved.items()) if src.name != f"art{n}.txt"]
orphans = [f for f in files if f not in claimed]

print(f"CSV rows: {len(df)}   files on disk: {len(files)}\n")
print(f"{len(moves)} file(s) to move:")
for src, dst, title in moves:
    print(f"  {src.name:11} -> {dst.name:11}  {title[:52]}")
print(f"\n{len(orphans)} orphan(s) (no longer in the CSV) -> articles/_archive/:")
for f in orphans:
    print(f"  {f.name:11} {(titles.get(f) or '')[:52]}")
print(f"\n{len(missing)} row(s) with no file — main() will scrape these:")
for n, t in missing:
    print(f"  row {n}: {t[:60]}")

if APPLY:
    archive_dir.mkdir(exist_ok=True)
    for f in orphans:
        shutil.move(str(f), str(archive_dir / f.name))
    # stage first so a chain like art11->art10, art12->art11 cannot clobber itself
    staging = art_dir / "_staging"
    staging.mkdir(exist_ok=True)
    for src, dst, _ in moves:
        shutil.move(str(src), str(staging / dst.name))
    for f in staging.glob("art*.txt"):
        shutil.move(str(f), str(art_dir / f.name))
    staging.rmdir()
    print(f"\nAPPLIED: {len(moves)} moved, {len(orphans)} archived.")
else:
    print("\n(dry run — set APPLY = True to execute)")
